## Cell 1 — Setup & Download

In [ ]:
# SETUP & DOWNLOAD
from google.colab import drive
drive.mount('/content/drive')

import os, urllib.request, sys

DRIVE_DIR = '/content/drive/MyDrive/CinC2017'
os.makedirs(DRIVE_DIR, exist_ok=True)
DATASET_ZIP = f"{DRIVE_DIR}/training2017.zip"

def progress_bar(count, block_size, total_size):
    if total_size > 0:
        percent = min(int(count * block_size * 100 / total_size), 100)
        sys.stdout.write(f"\rDownloading Dataset: {percent}% ")
        sys.stdout.flush()

if not os.path.exists(DATASET_ZIP):
    print("Fetching CinC 2017 Dataset to Google Drive...")
    url = "https://physionet.org/files/challenge-2017/1.0.0/training2017.zip"
    urllib.request.urlretrieve(url, DATASET_ZIP, reporthook=progress_bar)
    print("\nDownload complete and saved permanently to Drive.")
else:
    print("Dataset already exists in Google Drive. Skipping download.")

print("Extracting to fast local Colab storage...")
!cp {DATASET_ZIP} /content/training2017.zip
!unzip -q /content/training2017.zip -d /content/dataset
print("Done.")

Mounted at /content/drive
Dataset already exists in Google Drive. Skipping download.
Extracting to fast local Colab storage...
Done.


## Cell 2 — Data Preprocessing (DWT + Normalisation + Split)

In [ ]:
# DATA PREPROCESSING – DWT, NORMALISATION, STRATIFIED SPLIT
import pywt
import torch
import numpy as np
import scipy.io as sio
import csv
import os
import sys
from torch.utils.data import TensorDataset, DataLoader, Dataset
from sklearn.model_selection import StratifiedShuffleSplit

DATA_DIR      = '/content/dataset/training2017'
CSV_FILE      = os.path.join(DATA_DIR, 'REFERENCE.csv')
TARGET_LENGTH = 18000
LABEL_MAP     = {'N': 0, 'A': 1, 'O': 2, '~': 3}
LABEL_NAMES   = ['normal', 'af', 'other', 'noise']

def apply_dwt(signal_1d):
    coeffs = pywt.wavedec(signal_1d, 'db2', level=4, mode='zero')
    return np.stack([coeffs[0], coeffs[1]], axis=0)

print("Parsing REFERENCE.csv and loading .mat files...")
processed_data, labels = [], []

with open(CSV_FILE, 'r') as f:
    rows = list(csv.reader(f))
    total_files = len(rows)
    for i, row in enumerate(rows):
        record_name, label_str = row[0], row[1]
        mat_path = os.path.join(DATA_DIR, f"{record_name}.mat")
        raw_signal = sio.loadmat(mat_path)['val'][0].astype(np.float64)

        if len(raw_signal) < TARGET_LENGTH:
            raw_signal = np.pad(raw_signal, (0, TARGET_LENGTH - len(raw_signal)), 'constant')
        else:
            raw_signal = raw_signal[:TARGET_LENGTH]

        processed_data.append(apply_dwt(raw_signal))
        labels.append(LABEL_MAP[label_str])

        if (i + 1) % 500 == 0 or (i + 1) == total_files:
            sys.stdout.write(f"\rProcessed {i + 1}/{total_files} records...")
            sys.stdout.flush()

print("\nConverting to tensors...")
X = torch.tensor(np.array(processed_data), dtype=torch.float32)
y = torch.tensor(labels, dtype=torch.long)

# Per-sample, per-channel normalisation
mean = X.mean(dim=(1, 2), keepdim=True)
std  = X.std(dim=(1, 2),  keepdim=True) + 1e-8
X    = (X - mean) / std

# Stratified 80/20 split
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(sss.split(X, y))

print(f"Train: {len(train_idx)}  Test: {len(test_idx)}  Input shape: {X[0].shape}")

Parsing REFERENCE.csv and loading .mat files...
Processed 8528/8528 records...
Converting to tensors...
Train: 6822  Test: 1706  Input shape: torch.Size([2, 1127])


## Cell 3 — Model Architecture

In [ ]:
# PAPER-ACCURATE CNN ARCHITECTURE (Table I, Loh et al. ASAP 2020)
import torch.nn as nn

class ECGModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 10, kernel_size=(2, 5), bias=False)
        self.bn1   = nn.BatchNorm2d(10)
        self.pool1 = nn.MaxPool2d(kernel_size=(1, 3), stride=(1, 3))

        self.conv2 = nn.Conv2d(10, 24, kernel_size=(1, 5), bias=False)
        self.bn2   = nn.BatchNorm2d(24)
        self.pool2 = nn.MaxPool2d(kernel_size=(1, 3), stride=(1, 3))

        self.conv3 = nn.Conv2d(24, 24, kernel_size=(1, 5), bias=False)
        self.bn3   = nn.BatchNorm2d(24)
        self.pool3 = nn.MaxPool2d(kernel_size=(1, 3), stride=(1, 3))

        self.conv4 = nn.Conv2d(24, 24, kernel_size=(1, 5), bias=False)
        self.bn4   = nn.BatchNorm2d(24)
        self.pool4 = nn.MaxPool2d(kernel_size=(1, 3), stride=(1, 3))

        self.fc      = nn.Linear(264, 4)
        self.relu    = nn.ReLU()
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
      x = x.unsqueeze(1)
      x = torch.clamp(self.pool1(self.relu(self.bn1(self.conv1(x)))), -8.0, 7.99609375)
      x = torch.clamp(self.pool2(self.relu(self.bn2(self.conv2(x)))), -8.0, 7.99609375)
      x = torch.clamp(self.pool3(self.relu(self.bn3(self.conv3(x)))), -8.0, 7.99609375)
      x = torch.clamp(self.pool4(self.relu(self.bn4(self.conv4(x)))), -8.0, 7.99609375)
      x = x.view(x.size(0), -1)
      x = self.dropout(x)
      return self.fc(x)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
model = ECGModel().to(device)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

Device: cuda
Trainable parameters: 8284


## Cell 4 — Training

In [ ]:
# FP32 TRAINING — 5-fold stratified CV, 10 random seeds per fold
import torch.optim as optim
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

class AugmentedDataset(Dataset):
    def __init__(self, X, y, augment=True):
        self.X, self.y, self.augment = X, y, augment
    def __len__(self): return len(self.X)
    def __getitem__(self, idx):
        x = self.X[idx].clone()
        if self.augment:
            if np.random.rand() > 0.5: x = -x
            x = x + np.random.uniform(-0.2, 0.2) * x.std()
            if np.random.rand() > 0.7: x = x * np.random.uniform(0.8, 1.2)
        return x, self.y[idx]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
best_global_f1    = 0.0
best_global_state = None

for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f'\n=== Fold {fold+1}/5 ===')
    best_fold_f1    = 0.0
    best_fold_state = None

    for seed in range(3):
        torch.manual_seed(seed)
        np.random.seed(seed)

        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]

        class_counts  = torch.bincount(y_tr)
        class_weights = (len(y_tr) / (len(class_counts) * class_counts.float())).to(device)

        train_loader = DataLoader(AugmentedDataset(X_tr, y_tr, augment=True),  batch_size=32, shuffle=True)
        val_loader   = DataLoader(AugmentedDataset(X_val, y_val, augment=False), batch_size=32, shuffle=False)

        model     = ECGModel().to(device)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
        optimizer = optim.Adam(model.parameters(), lr=3e-4, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=5, factor=0.5)

        for epoch in range(80):
            model.train()
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                loss = criterion(model(inputs), labels)
                loss.backward()
                optimizer.step()

            model.eval()
            all_preds, all_targets = [], []
            with torch.no_grad():
                for inputs, labels in val_loader:
                    preds = model(inputs.to(device)).argmax(1).cpu().numpy()
                    all_preds.extend(preds)
                    all_targets.extend(labels.numpy())

            cinc_f1 = f1_score(all_targets, all_preds, labels=[0,1,2], average='macro')
            scheduler.step(cinc_f1)

            if cinc_f1 > best_fold_f1:
                best_fold_f1    = cinc_f1
                best_fold_state = {k: v.clone() for k, v in model.state_dict().items()}

        print(f'  Seed {seed}: best CinC F1 = {best_fold_f1:.4f}')

    print(f'Fold {fold+1} best: {best_fold_f1:.4f}')
    if best_fold_f1 > best_global_f1:
        best_global_f1    = best_fold_f1
        best_global_state = best_fold_state

print(f'\nBest global CinC F1: {best_global_f1:.4f}')
model.load_state_dict(best_global_state)
torch.save(model.state_dict(), '/content/drive/MyDrive/best_model.pth')
print('Saved best_model.pth')


=== Fold 1/5 ===
  Seed 0: best CinC F1 = 0.7269
  Seed 1: best CinC F1 = 0.7269
  Seed 2: best CinC F1 = 0.7269
Fold 1 best: 0.7269

=== Fold 2/5 ===
  Seed 0: best CinC F1 = 0.7149
  Seed 1: best CinC F1 = 0.7181
  Seed 2: best CinC F1 = 0.7181
Fold 2 best: 0.7181

=== Fold 3/5 ===
  Seed 0: best CinC F1 = 0.7037
  Seed 1: best CinC F1 = 0.7037
  Seed 2: best CinC F1 = 0.7037
Fold 3 best: 0.7037

=== Fold 4/5 ===
  Seed 0: best CinC F1 = 0.7211
  Seed 1: best CinC F1 = 0.7464
  Seed 2: best CinC F1 = 0.7464
Fold 4 best: 0.7464

=== Fold 5/5 ===
  Seed 0: best CinC F1 = 0.7061
  Seed 1: best CinC F1 = 0.7201
  Seed 2: best CinC F1 = 0.7201
Fold 5 best: 0.7201

Best global CinC F1: 0.7464
Saved best_model.pth


## Cell 5 — BN Folding + Hardware-Accurate Evaluation

In [ ]:
# BN FOLDING + Q4.8 HARDWARE-ACCURATE INFERENCE + FULL TEST SET EVALUATION
from sklearn.metrics import classification_report, f1_score
import torch.nn.functional as F

# ── 1. BN Folding ──────────────────────────────────────────────────────────
def fold_bn(conv_w, bn_mean, bn_var, bn_gamma, bn_beta, eps=1e-5):
    std   = np.sqrt(bn_var + eps)
    scale = bn_gamma / std
    for _ in range(conv_w.ndim - 1):
        scale = scale[:, np.newaxis]
    return conv_w * scale, bn_beta - bn_mean * (bn_gamma / std)

model.eval()
model = model.cpu()
sd = {k: v.detach().numpy() for k, v in model.state_dict().items()}

w1, b1 = fold_bn(sd['conv1.weight'], sd['bn1.running_mean'], sd['bn1.running_var'], sd['bn1.weight'], sd['bn1.bias'])
w2, b2 = fold_bn(sd['conv2.weight'], sd['bn2.running_mean'], sd['bn2.running_var'], sd['bn2.weight'], sd['bn2.bias'])
w3, b3 = fold_bn(sd['conv3.weight'], sd['bn3.running_mean'], sd['bn3.running_var'], sd['bn3.weight'], sd['bn3.bias'])
w4, b4 = fold_bn(sd['conv4.weight'], sd['bn4.running_mean'], sd['bn4.running_var'], sd['bn4.weight'], sd['bn4.bias'])
wd, bd = sd['fc.weight'], sd['fc.bias']
print('BN folding done.')

# ── 2. Hardware-accurate inference ─────────────────────────────────────────
def quantize_tensor(x):
    return torch.clamp(torch.round(x * 256) / 256, -8.0, 7.99609375)

w1t = torch.tensor(w1, dtype=torch.float32)
b1t = torch.tensor(b1, dtype=torch.float32)
w2t = torch.tensor(w2, dtype=torch.float32)
b2t = torch.tensor(b2, dtype=torch.float32)
w3t = torch.tensor(w3, dtype=torch.float32)
b3t = torch.tensor(b3, dtype=torch.float32)
w4t = torch.tensor(w4, dtype=torch.float32)
b4t = torch.tensor(b4, dtype=torch.float32)
wdt = torch.tensor(wd, dtype=torch.float32)
bdt = torch.tensor(bd, dtype=torch.float32)

def hw_inference(x):
    x = x.unsqueeze(1)
    x = quantize_tensor(x)
    x = F.conv2d(x, quantize_tensor(w1t), quantize_tensor(b1t))
    x = quantize_tensor(F.relu(x) / 2.0)  # layer 1 max=18 → /2 → 9, just over range
    x = F.max_pool2d(x, kernel_size=(1,3), stride=(1,3))

    x = F.conv2d(x, quantize_tensor(w2t), quantize_tensor(b2t))
    x = F.relu(x)  # no scaling, max=15 but let it flow
    x = F.max_pool2d(x, kernel_size=(1,3), stride=(1,3))

    x = F.conv2d(x, quantize_tensor(w3t), quantize_tensor(b3t))
    x = F.relu(x)
    x = F.max_pool2d(x, kernel_size=(1,3), stride=(1,3))

    x = F.conv2d(x, quantize_tensor(w4t), quantize_tensor(b4t))
    x = F.relu(x)
    x = F.max_pool2d(x, kernel_size=(1,3), stride=(1,3))

    x = x.view(x.size(0), -1)
    return F.linear(x, quantize_tensor(wdt), quantize_tensor(bdt))

# ── 3. Evaluate on full test set ───────────────────────────────────────────
test_ds     = AugmentedDataset(X[test_idx], y[test_idx], augment=False)  # add
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)           # add
all_preds, all_targets = [], []
all_preds, all_targets = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        outputs = hw_inference(inputs.cpu())
        all_preds.extend(outputs.argmax(1).numpy())
        all_targets.extend(labels.numpy())

CLASS_NAMES = ['Normal', 'AF', 'Other', 'Noise']
print("=" * 50)
print(" HARDWARE-ACCURATE RESULTS (Q4.8, BN FOLDED) ")
print("=" * 50)
print(classification_report(all_targets, all_preds, target_names=CLASS_NAMES))
cinc_f1 = f1_score(all_targets, all_preds, labels=[0,1,2], average='macro')
print(f"CinC F1 (N/AF/O macro): {cinc_f1:.4f}")
print(f"Weighted F1:            {f1_score(all_targets, all_preds, average='weighted'):.4f}")

BN folding done.
 HARDWARE-ACCURATE RESULTS (Q4.8, BN FOLDED) 
              precision    recall  f1-score   support

      Normal       0.84      0.91      0.87      1010
          AF       0.67      0.64      0.66       148
       Other       0.77      0.56      0.65       491
       Noise       0.54      0.93      0.68        57

    accuracy                           0.79      1706
   macro avg       0.70      0.76      0.71      1706
weighted avg       0.79      0.79      0.78      1706

CinC F1 (N/AF/O macro): 0.7264
Weighted F1:            0.7831


In [ ]:
# PROFILE ACTIVATION RANGES PER LAYER
import torch.nn.functional as F

layer_outputs = {1: [], 2: [], 3: [], 4: []}

def profile_inference(x):
    x = x.unsqueeze(1)
    x = quantize_tensor(x)
    x = F.conv2d(x, quantize_tensor(w1t), quantize_tensor(b1t))
    x = F.relu(x)
    x = F.max_pool2d(x, kernel_size=(1,3), stride=(1,3))
    layer_outputs[1].append(x.abs().max().item())

    x = F.conv2d(x, quantize_tensor(w2t), quantize_tensor(b2t))
    x = F.relu(x)
    x = F.max_pool2d(x, kernel_size=(1,3), stride=(1,3))
    layer_outputs[2].append(x.abs().max().item())

    x = F.conv2d(x, quantize_tensor(w3t), quantize_tensor(b3t))
    x = F.relu(x)
    x = F.max_pool2d(x, kernel_size=(1,3), stride=(1,3))
    layer_outputs[3].append(x.abs().max().item())

    x = F.conv2d(x, quantize_tensor(w4t), quantize_tensor(b4t))
    x = F.relu(x)
    x = F.max_pool2d(x, kernel_size=(1,3), stride=(1,3))
    layer_outputs[4].append(x.abs().max().item())
    return x

with torch.no_grad():
    for inputs, labels in test_loader:
        for i in range(inputs.shape[0]):
            profile_inference(inputs[i:i+1].cpu())

import numpy as np
for l in range(1, 5):
    vals = layer_outputs[l]
    print(f'Layer {l}: max={max(vals):.2f}  mean={np.mean(vals):.2f}  p99={np.percentile(vals,99):.2f}')

Layer 1: max=17.83  mean=7.68  p99=13.84
Layer 2: max=14.19  mean=5.15  p99=9.70
Layer 3: max=11.99  mean=4.22  p99=9.20
Layer 4: max=10.35  mean=4.03  p99=8.08


## Cell 6 — Weight Export (weights.hex)

In [ ]:
SCALE   = 256       # same — 8 fractional bits
MAX_VAL =  32767    # 2^15 - 1
MIN_VAL = -32768    # -2^15

def to_q88(x):
    q = int(round(float(x) * SCALE))
    return max(MIN_VAL, min(MAX_VAL, q))

def to_hex16(v):
    if v < 0: v += 65536
    return f'{v:04X}'

all_vals = []

# Conv1 — [10, 1, 2, 5]
for oc in range(w1.shape[0]):
    for ic in range(w1.shape[1]):
        for row in range(w1.shape[2]):
            for tap in range(w1.shape[3]):
                all_vals.append(to_q88(w1[oc, ic, row, tap]))
for v in b1: all_vals.append(to_q88(v))

# Conv2 — [24, 10, 1, 5]
for oc in range(w2.shape[0]):
    for ic in range(w2.shape[1]):
        for tap in range(w2.shape[3]):
            all_vals.append(to_q88(w2[oc, ic, 0, tap]))
for v in b2: all_vals.append(to_q88(v))

# Conv3 — [24, 24, 1, 5]
for oc in range(w3.shape[0]):
    for ic in range(w3.shape[1]):
        for tap in range(w3.shape[3]):
            all_vals.append(to_q88(w3[oc, ic, 0, tap]))
for v in b3: all_vals.append(to_q88(v))

# Conv4 — [24, 24, 1, 5]
for oc in range(w4.shape[0]):
    for ic in range(w4.shape[1]):
        for tap in range(w4.shape[3]):
            all_vals.append(to_q88(w4[oc, ic, 0, tap]))
for v in b4: all_vals.append(to_q88(v))

# Dense — [4, 264]
for oc in range(wd.shape[0]):
    for ic in range(wd.shape[1]):
        all_vals.append(to_q88(wd[oc, ic]))
for v in bd: all_vals.append(to_q88(v))

print(f'Total values: {len(all_vals)}  (expected 8202)')
if len(all_vals) < 8202:
    all_vals.extend([0] * (8202 - len(all_vals)))
else:
    all_vals = all_vals[:8202]

with open('weights.hex', 'w') as f:
    for v in all_vals:
        f.write(to_hex16(v) + '\n')
print('weights.hex written.')

from google.colab import files
files.download('weights.hex')

Total values: 8202  (expected 8202)
weights.hex written.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Cell 7 — ECG Stimulus Hex Export (25 per class)

In [ ]:
# ECG STIMULUS HEX EXPORT — Top 25 confident correct predictions per class
# Each class file = 6 * 18000 = 108000 samples (set WINS_PER_CLASS=25 in TB)
WINDOW = 18000
TOP_N  = 6

def sig_to_hex12(sig):
    sig = sig - sig.mean()
    peak = np.abs(sig).max()
    if peak > 1e-6:
        sig = sig / peak * 2000.0
    lines = []
    for s in sig:
        v = int(round(float(s)))
        v = max(-2048, min(2047, v))
        if v < 0: v += 4096
        lines.append(f'{v:03X}')
    return lines

with open(CSV_FILE) as f:
    all_recs = [(r[0], LABEL_MAP[r[1]]) for r in csv.reader(f) if r[1] in LABEL_MAP]

best = {0: [], 1: [], 2: [], 3: []}

model.eval()
for rec_name, true_lbl in all_recs:
    try:
        sig  = sio.loadmat(os.path.join(DATA_DIR, f'{rec_name}.mat'))['val'][0].astype(np.float32)
        if len(sig) < WINDOW: continue
        sig_w = sig[:WINDOW]

        sig_n = sig_w - sig_w.mean()
        std   = sig_n.std()
        if std < 1e-6: continue
        sig_n = sig_n / std

        coeffs  = pywt.wavedec(sig_n, 'db2', level=4, mode='zero')
        A4, D4  = coeffs[0], coeffs[1]
        min_len = min(len(A4), len(D4))
        feat    = np.stack([A4[:min_len], D4[:min_len]], axis=0)

        x = torch.tensor(feat, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            probs = torch.softmax(hw_inference(x), dim=1)[0].numpy()
            pred  = int(probs.argmax())

        if pred == true_lbl and probs[pred] > 0.85:
            best[true_lbl].append((probs[pred], sig_w))
    except: pass

for lbl, name in enumerate(LABEL_NAMES):
    sorted_sigs = [s for _, __, s in sorted([(conf, i, sig) for i, (conf, sig) in enumerate(best[lbl])], reverse=True)[:TOP_N]]
    print(f'{name}: {len(sorted_sigs)} recordings found')
    with open(f'ecg_{name}.hex', 'w') as f:
        for sig in sorted_sigs:
            f.write('\n'.join(sig_to_hex12(sig)) + '\n')
    files.download(f'ecg_{name}.hex')

print('Done.')

normal: 6 recordings found


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

af: 6 recordings found


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

noise: 6 recordings found


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done.


## Cell 8 — Verify Hex Files Against hw_inference

In [ ]:
# VERIFY: Load hex files back and confirm hw_inference predicts correctly
CLASS_NAMES_TITLE = ['Normal', 'AF', 'Other', 'Noise']

def load_hex_as_signal(hex_path, win=18000):
    with open(hex_path) as f:
        lines = [l.strip() for l in f if l.strip()]
    results = []
    for w in range(len(lines) // win):
        chunk = lines[w*win:(w+1)*win]
        samples = []
        for l in chunk:
            v = int(l, 16)
            if v >= 2048: v -= 4096
            samples.append(v)
        sig = np.array(samples, dtype=np.float32) / 2000.0
        coeffs = pywt.wavedec(sig, 'db2', level=4, mode='zero')
        A4, D4 = coeffs[0], coeffs[1]
        min_len = min(len(A4), len(D4))
        feat = np.stack([A4[:min_len], D4[:min_len]], axis=0)
        results.append(feat)
    return results

correct, total = 0, 0
for lbl, name in enumerate(LABEL_NAMES):
    feats = load_hex_as_signal(f'ecg_{name}.hex')
    for feat in feats:
        x = torch.tensor(feat, dtype=torch.float32).unsqueeze(0)
        with torch.no_grad():
            pred = hw_inference(x).argmax(1).item()
        correct += (pred == lbl)
        total   += 1

print(f"Python hw_inference accuracy on hex files: {correct}/{total} ({100*correct/total:.1f}%)")

Python hw_inference accuracy on hex files: 23/24 (95.8%)


In [ ]:
with open('weights.hex') as f:
    lines = f.readlines()
print("weights.hex[0] =", lines[0].strip())

weights.hex[0] = 075


In [ ]:
for i, feat in enumerate(load_hex_as_signal('ecg_af.hex')):
    x = torch.tensor(feat, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        logits = hw_inference(x)[0].numpy()
        pred = int(logits.argmax())
    print(f'af win {i+1}: pred={LABEL_NAMES[pred]} logits={logits.round().astype(int).tolist()}')

af win 1: pred=af logits=[-3, 2, 1, -12]
af win 2: pred=af logits=[-1, 3, 2, -15]
af win 3: pred=other logits=[0, 3, 3, -17]
af win 4: pred=af logits=[-1, 3, 2, -16]
af win 5: pred=af logits=[-2, 4, 2, -18]
af win 6: pred=af logits=[-1, 3, 2, -16]
